# Background aerosol, model response and online sources: CTRL_PD vs CTRL_FUT

**Question.** What aerosol, chemistry and natural-source background does the model produce in the PD and FUT runs, and how different is it?

**Runs.** CTRL_PD `NF2000norbc_tropstratchem_nudg_ctrl_f19_f19-20261029` and CTRL_FUT `NF2100ssp585norbc_tropstratchem_nudg_ctrl_f19_f19-20261028` (new chain), **2001–2025** (first 12 months dropped as spin-up). Both are nudged to the same PD winds.

**Companion notebook:** `background_aerosol_pd_fut_prescribed.ipynb` — the prescribed inputs each run reads (emission files of year 2000 / 2100 by sector, ocean DMS concentration, greenhouse-gas boundary condition) and what each compound does in the model. The LCC_FUT − CTRL_FUT version of this notebook is `background_aerosol_lcc_ctrl_fut_online.ipynb`.

**What this notebook shows.**
1. What the model makes of the inputs: AOD, sulfate burden, droplet number, surface SO₂, DMS, NOx, O₃, CO (CTRL_PD vs CTRL_FUT, 2001–2025), tables by region and maps.
2. Online sources: MEGAN BVOC, dust, sea salt, ocean DMS flux, lightning NOx: tables, maps, global totals (bar chart, 2001–2025 mean) and yearly time series (§2.1), as for the prescribed emissions.
3. Summary.
4. Appendix: the same maps, global.

**Maps** show the **Arctic** (north of 45°N, polar view; dashed circle = Arctic Circle, 66.5°N). The global versions of every map are in the Appendix (files in `figures/appendix-global/`).


In [ ]:
import os
os.environ.setdefault('HDF5_USE_FILE_LOCKING', 'FALSE')  # before netCDF4/xarray import
import re
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
import numpy as np
import pandas as pd
import xarray as xr
import netCDF4
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.path as mpath
import cartopy.crs as ccrs

CASEDIR = Path.home() / 'cases/BRL_FRST_XPSN'
CASES = {'PD': 'NF2000norbc_tropstratchem_nudg_ctrl_f19_f19-20261029',
         'FUT': 'NF2100ssp585norbc_tropstratchem_nudg_ctrl_f19_f19-20261028'}
ALIAS = {'PD': 'CTRL_PD', 'FUT': 'CTRL_FUT'}
PROC = Path('../../data/processed-data/postprocessing')
PROC_YEARS = '2001-2025'
YEARS = list(range(int(PROC_YEARS[:4]), int(PROC_YEARS[-4:]) + 1))
YRS = f'{YEARS[0]}–{YEARS[-1]}'
DMS_DIR = Path('/cluster/shared/noresm/inputdata/noresm-only/atm/cam/camoslo')
OUT = Path('background_aerosol_pd_fut_online_outputs')
CACHE, FIG, TAB = OUT / 'cache', OUT / 'figures', OUT / 'tables'  # cache = regenerable totals and maps
FIG_GLOBAL = FIG / 'appendix-global'  # global maps (Appendix)
for d in (CACHE, FIG, TAB, FIG_GLOBAL):
    d.mkdir(parents=True, exist_ok=True)
REBUILD_CACHE = False
N_WORKERS = 8
LAT_MAP = 45      # Arctic maps: north of this latitude
LAT_BOX = 40      # regional totals (bars, time series): north of this latitude
APPENDIX = []     # map calls, replayed globally in the Appendix

NA, R_EARTH = 6.02214076e23, 6.371e6
DPM = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])  # noleap
# project palette, in order; gold = total; PD = navy, FUT = red
TEAL, NAVY, PLUM, RED, ORANGE, GOLD, GREY = '#5D9A92', '#2C3990', '#9A4A84', '#BC423E', '#DE7A3A', '#EBC45C', '#CCCCCC'
PCOL = {'PD': NAVY, 'FUT': RED}


In [ ]:
def cell_area(lat, nlon):
    lat = np.asarray(lat, float)
    edges = np.clip(np.concatenate([[-90], 0.5 * (lat[1:] + lat[:-1]), [90]]), -90, 90)
    band = R_EARTH**2 * np.deg2rad(360 / nlon) * np.diff(np.sin(np.deg2rad(edges)))
    return np.repeat(band[:, None], nlon, axis=1)  # m2

# model grid (f19) from the processed files
with xr.open_dataset(PROC / f'{ALIAS["PD"]}_atm_AEROPT_{PROC_YEARS}.nc') as ds:
    AREA = xr.DataArray(cell_area(ds.lat, ds.lon.size), dims=('lat', 'lon'), coords=dict(lat=ds.lat.values, lon=ds.lon.values))
assert abs(AREA.sum() / (4 * np.pi * R_EARTH**2) - 1) < 1e-6

def nice_levels(vmax, decades=3):
    """1-2-5 levels from the first one >= vmax down by `decades` decades."""
    e = np.floor(np.log10(vmax))
    cand = np.array([m * 10.0**k for k in np.arange(e - decades - 1, e + 2) for m in (1, 2, 5)])
    hi = cand[cand >= vmax * 0.999][0]
    return cand[(cand <= hi) & (cand >= hi / 10**decades * 0.999)]

def cyclic(da):
    return xr.concat([da, da.isel(lon=0).assign_coords(lon=360.)], 'lon')

def map_axes(ncols, region, width=16):
    """Robinson (global) or north-polar stereographic north of LAT_MAP (arctic)."""
    if region == 'global':
        return plt.subplots(1, ncols, figsize=(width, 3.4), subplot_kw=dict(projection=ccrs.Robinson()))
    fig, axs = plt.subplots(1, ncols, figsize=(width, 6.2), subplot_kw=dict(projection=ccrs.NorthPolarStereo()))
    th = np.linspace(0, 2 * np.pi, 200)
    for ax in np.atleast_1d(axs):
        ax.set_extent([-180, 180, LAT_MAP, 90], ccrs.PlateCarree())
        ax.set_boundary(mpath.Path(np.c_[np.sin(th), np.cos(th)] * 0.5 + 0.5), transform=ax.transAxes)
    return fig, axs

def in_map(da, region):
    """Values inside the plotted domain (for the colour levels)."""
    return da if region == 'global' else da.where(da.lat >= LAT_MAP)

def white_zero(cmap, levels, extend='both'):
    """Same colours per bin as BoundaryNorm(levels, cmap.N, extend), except the bin that contains 0 is pure white."""
    cm = plt.get_cmap(cmap) if isinstance(cmap, str) else cmap
    levels = np.asarray(levels, float)
    off, n_ext = {'neither': (0, 0), 'min': (1, 1), 'max': (0, 1), 'both': (1, 2)}[extend]
    nreg = len(levels) - 1 + n_ext
    cols = cm(((cm.N - 1) / (nreg - 1) * np.arange(nreg)).astype(int))  # BoundaryNorm's bin -> colour index
    for i in range(len(levels) - 1):
        if levels[i] < 0 < levels[i + 1]:
            cols[i + off] = (1, 1, 1, 1)
    return mcolors.ListedColormap(cols), mcolors.BoundaryNorm(levels, nreg, extend=extend)

def draw(ax, da, cmap, norm):
    da = cyclic(da)
    da = da.where(abs(da) > 1e-6 * np.abs(norm.boundaries).max())  # (near-)zero -> not drawn = white
    im = ax.pcolormesh(da.lon, da.lat, da, cmap=cmap, norm=norm, transform=ccrs.PlateCarree(), shading='nearest')
    ax.coastlines(lw=0.4)
    if ax.projection.__class__.__name__ == 'NorthPolarStereo':  # Arctic Circle
        ax.plot(np.linspace(0, 360, 361), np.full(361, 66.5), ls='--', lw=0.6, color='k', transform=ccrs.PlateCarree())
    return im

def save_map(fig, fname, region):
    fig.savefig((FIG if region == 'arctic' else FIG_GLOBAL) / fname, dpi=200, bbox_inches='tight')
    plt.show()

def region_note(region):
    return f'Arctic, north of {LAT_MAP}°N (dashed = 66.5°N)' if region == 'arctic' else 'global'

def triptych(fields, title, unit, fname, decades=3, labels=None, region='arctic'):
    """fields = {'PD': da, 'FUT': da}; log-scale absolute, symmetric log-spaced difference.
    Colour levels from the plotted domain. Arctic calls are queued for the global Appendix."""
    if region == 'arctic':
        APPENDIX.append((triptych, (fields, title, unit, fname), dict(decades=decades, labels=labels)))
    pd_, fu = fields['PD'], fields['FUT']
    diff = fu - pd_
    vmax = float(np.nanpercentile(np.r_[in_map(pd_, region).values.ravel(), in_map(fu, region).values.ravel()], 99.5))
    lv = nice_levels(vmax, decades)
    cm_abs = plt.get_cmap('YlOrBr').copy()
    cm_abs.set_under('white')
    norm_abs = mcolors.BoundaryNorm(lv, cm_abs.N, extend='max')
    dmax = float(np.nanpercentile(abs(in_map(diff, region).values), 99.5))
    dl = nice_levels(dmax, decades - 1)
    dlev = np.r_[-dl[::-1], dl]  # the central bin (-dl[0], dl[0]) is white
    cm_d, norm_d = white_zero('RdBu_r', dlev)
    fig, axs = map_axes(3, region)
    labels = labels or {'PD': f'PD ({CYCLE["PD"]})', 'FUT': f'FUT ({CYCLE["FUT"]})'}
    for ax, (lab, da) in zip(axs[:2], [(labels['PD'], pd_), (labels['FUT'], fu)]):
        im = draw(ax, da, cm_abs, norm_abs)
        ax.set_title(lab)
    fig.colorbar(im, ax=axs[:2], orientation='horizontal', shrink=0.6, pad=0.04, ticks=lv, label=unit,
                 format=lambda x, _: f'{x:g}')
    im = draw(axs[2], diff, cm_d, norm_d)
    axs[2].set_title('FUT − PD')
    cb = fig.colorbar(im, ax=axs[2], orientation='horizontal', shrink=0.95, pad=0.04, ticks=dlev[::2] if len(dlev) > 12 else dlev,
                      label=unit, format=lambda x, _: f'{x:g}')
    cb.ax.tick_params(labelsize=7, rotation=45)
    fig.suptitle(f'{title}\n{region_note(region)}', y=1.06)
    save_map(fig, fname, region)


## 1. What the model makes of it: CTRL_PD vs CTRL_FUT, 2001–2025

Processed monthly files, 2001–2025 (first 12 months already dropped), means weighted by days per month. Surface = lowest model level. These are model results, so they include the online sources (sea salt, dust, DMS, MEGAN) and the FUT climate (SST, CO₂), not only the prescribed emissions. Both runs are nudged to the same PD winds.

**Regions.** global, NH, SH, **46–82°N** and Arctic (>66.5°N). 46–82°N is the **forest-expansion band**: the only latitudes where the LCC land cover (`PCT_NAT_PFT`) differs from CTRL (46.4–82.4°N, from a diff of the two surfdata files). It is the same band as in the other diagnostics, so the numbers can be compared. It is **not** a 45–90°N box: the ocean and Greenland north of 82°N are left out because their land cover does not change; the Arctic row covers the pole. Maps: Arctic (north of 45°N); the map titles give the Arctic (>66.5°N) mean. Global maps in the Appendix.

In [ ]:
MODEL_VARS = {  # name: (category, variable(s), surface?, factor, unit)
    'AOD 550 nm': ('AEROPT', ['DOD550'], False, 1, '1'),
    'absorption AOD 550 nm': ('AEROPT', ['ABS550'], False, 1, '1'),
    'sulfate burden': ('SULFATE', ['cb_SULFATE'], False, 1e6, 'mg m⁻²'),
    'droplet column': ('CLOUDPROP', ['CDNUMC'], False, 1e-10, '10¹⁰ m⁻²'),
    'surface SO₂': ('SULFATE', ['SO2'], True, 1e12, 'ppt'),
    'surface DMS': ('SULFATE', ['DMS'], True, 1e12, 'ppt'),
    'surface NOx': ('CHEM', ['NO', 'NO2'], True, 1e12, 'ppt'),
    'surface O₃': ('CHEM', ['O3'], True, 1e9, 'ppb'),
    'surface CO': ('CHEM', ['CO'], True, 1e9, 'ppb'),
    'O₃ column': ('CHEM', ['cb_O3'], False, 1e3, 'g m⁻²'),
}

def load_model(period):
    out, files = {}, {}
    for name, (cat, vs, sfc, fac, unit) in MODEL_VARS.items():
        f = PROC / f'{ALIAS[period]}_atm_{cat}_{PROC_YEARS}.nc'
        if f not in files:
            files[f] = xr.open_dataset(f)
        ds = files[f]
        assert ds.attrs['casename'] == CASES[period], (f, ds.attrs['casename'])
        assert ds.time.encoding.get('calendar') == 'noleap'
        da = sum(ds[v].isel(lev=-1) if sfc else ds[v] for v in vs)
        days = (ds.time_bnds[:, 1] - ds.time_bnds[:, 0]).dt.days
        out[name] = ((da * days).sum('time') / days.sum()).load() * fac
    assert all(float(ds.lev[-1]) > 900 for ds in files.values() if 'lev' in ds.dims)
    return out

MODEL = {p: load_model(p) for p in CASES}
for p in CASES:
    for k in MODEL[p]:
        assert np.allclose(MODEL[p][k].lat, AREA.lat) and np.allclose(MODEL[p][k].lon, AREA.lon)
        MODEL[p][k] = MODEL[p][k].assign_coords(lat=AREA.lat, lon=AREA.lon)

LAT = AREA.lat.broadcast_like(AREA)
REGIONS = {'global': AREA, 'NH': AREA * (LAT > 0), 'SH': AREA * (LAT < 0), '46–82°N': AREA * ((LAT >= 46) & (LAT <= 82)),
           'Arctic >66.5°N': AREA * (LAT > 66.5)}
rows = []
for name, (_, _, _, _, unit) in MODEL_VARS.items():
    for r, w in REGIONS.items():
        v = {p: float((MODEL[p][name] * w).sum() / w.sum()) for p in CASES}
        rows.append(dict(variable=name, unit=unit, region=r, PD=v['PD'], FUT=v['FUT'], **{'FUT − PD': v['FUT'] - v['PD'],
                         'change %': 100 * (v['FUT'] / v['PD'] - 1)}))
MODEL_TAB = pd.DataFrame(rows).set_index(['variable', 'unit', 'region'])
MODEL_TAB.to_csv(TAB / f'model_levels_ctrl_pd_fut_{PROC_YEARS}.csv')
with pd.option_context('display.float_format', '{:.3g}'.format, 'display.max_rows', 100):
    display(MODEL_TAB.xs('global', level='region'))
    display(MODEL_TAB.xs('46–82°N', level='region'))
    display(MODEL_TAB.xs('Arctic >66.5°N', level='region'))

In [ ]:
def triptych_lin(name, levels, dlevels, cmap='YlOrBr', region='arctic'):
    if region == 'arctic':
        APPENDIX.append((triptych_lin, (name, levels, dlevels), dict(cmap=cmap)))
    unit = MODEL_VARS[name][4]
    reg = 'Arctic >66.5°N' if region == 'arctic' else 'global'
    fig, axs = map_axes(3, region)
    norm = mcolors.BoundaryNorm(levels, plt.get_cmap(cmap).N, extend='max')
    g = MODEL_TAB.loc[(name, unit, reg)]
    rl = reg.split()[0]
    for ax, p in zip(axs[:2], CASES):
        im = draw(ax, MODEL[p][name], cmap, norm)
        ax.set_title(f'{ALIAS[p]}: {rl} {g[p]:.3g} {unit if unit != "1" else ""}')
    fig.colorbar(im, ax=axs[:2], orientation='horizontal', shrink=0.6, pad=0.04, ticks=levels, label=unit, format=lambda x, _: f'{x:g}')
    im = draw(axs[2], MODEL['FUT'][name] - MODEL['PD'][name], *white_zero('RdBu_r', dlevels))
    axs[2].set_title(f'FUT − PD: {rl} {g["FUT − PD"]:+.3g} ({g["change %"]:+.0f} %)')
    cb = fig.colorbar(im, ax=axs[2], orientation='horizontal', shrink=0.95, pad=0.04, ticks=dlevels, label=unit, format=lambda x, _: f'{x:g}')
    cb.ax.tick_params(labelsize=7, rotation=45)
    fig.suptitle(f'{name}, {YRS} mean\n{region_note(region)}', y=1.06)
    save_map(fig, f'model_map_{re.sub(r"[^A-Za-z0-9]+", "_", name.replace("₂", "2").replace("₃", "3")).strip("_")}.png', region)

sym = lambda *v: np.r_[-np.array(v[::-1]), np.array(v)]
triptych_lin('AOD 550 nm', [0, .025, .05, .075, .1, .15, .2, .3, .4, .5, .6], sym(.01, .02, .05, .1, .2))
triptych_lin('sulfate burden', [0, 1, 2, 3, 4, 5, 7.5, 10, 15, 20, 30], sym(.5, 1, 2, 5, 10, 20))
triptych_lin('droplet column', [0, 1, 2, 3, 4, 5, 6, 8, 10, 12], sym(.25, .5, 1, 2, 4))
triptych_lin('surface SO₂', [0, 10, 20, 50, 100, 200, 500, 1000, 2000, 5000], sym(10, 50, 100, 500, 1000, 5000))
triptych_lin('surface DMS', [0, 10, 20, 50, 100, 150, 200, 300, 400], sym(10, 20, 50, 100, 200))
triptych_lin('surface NOx', [0, 10, 20, 50, 100, 200, 500, 1000, 2000, 5000, 10000], sym(10, 50, 100, 500, 1000, 5000))
triptych_lin('surface O₃', [0, 10, 15, 20, 25, 30, 35, 40, 45, 50, 60, 70], sym(2, 5, 10, 15, 20, 30))

## 2. Online sources: MEGAN BVOC, dust, sea salt, ocean DMS flux, lightning NOx

These are computed by the model every time step, so they respond to the FUT climate (and, in the LCC runs, to the land cover). CTRL_PD vs CTRL_FUT, 2001–2025. How each one is obtained:

| Source | What drives it in the model | How it is computed here | Evidence |
|---|---|---|---|
| **MEGAN BVOC** (19 CAM species) | CLM vegetation, leaf area, temperature, light, soil moisture, CO₂ | CLM `MEG_<compound>` (processed `lnd_BVOC`) × `landfrac`, summed into CAM species with `megan_specifier` (`drv_flds_in`) | (a) model output; check vs CAM `MEG_ISOP`, `MEG_MTERP` below |
| **Dust** | CLM: wind, soil moisture, snow, vegetation cover | raw CLM `DSTFLXT` × `landfrac` × erodibility (`soil_erod_file`, 0 below 0.1) / `dust_emis_fact` × 1.15, as `oslo_aero_dust_emis` | (a) formula; the result is what CAM receives (b): no CAM dust-emission output to check |
| **Sea salt** (3 modes, number flux) | 10-m wind (∝ U¹⁰^3.74), open ocean = ocean × (1 − ice), SST | Salter et al. (2015) formula of `oslo_aero_seasalt.F90` on **monthly means** of `U10`, `OCNFRAC`, `ICEFRAC` and the prescribed SST | (b): the model uses each time step; monthly means underestimate the absolute flux (U³·⁷⁴ is convex), the FUT/PD ratio is more robust |
| **Ocean DMS flux** | seawater DMS × transfer velocity (∝ U10² × Schmidt-number term of SST) × open ocean | `oslo_aero_dms_emis` (HAMOCC branch) on monthly means | (b), same caveat |
| **Lightning NOx** | convective cloud-top height (Price & Rind), `lght_no_prd_factor` | global value CAM prints every time step in `atm.log` (`Global flash freq …, lightning NOx (TgN/y)`), averaged per year | (a) model log; global only (no map in the output) |

No CAM output of sea-salt, dust or DMS emission exists in these runs (`D550_SS` and `SFDMS` not in `fincl`). Both periods use the **same nudged winds**, so a change in sea salt or DMS flux comes from sea ice, SST and (DMS) seawater concentration.

In [ ]:
import gzip
ARCHIVES = [Path('/cluster/work/users/adelez/archive'), Path.home() / 'BRL-FRST-XPSN_archive']  # live archive first (has 2015–2025)
SEC_YR = 365 * 86400
MW_DMS, MW_S = 62.13, 32.06
# Salter et al. (2015) sea-salt number flux, 3 modes: coefficients of SST³, SST², SST, 1 (oslo_aero_seasalt.F90)
SS_COEF = np.array([[-5.2168e5, 3.31725e7, -6.95275e8, 1.0684e10],
                    [0.0,       7.374e5,  -2.4803e7,   7.7373e8],
                    [0.0,       1.4210e4,  1.4662e7,   1.7075e8]])

def read_nml(case, key, fname='atm_in'):
    txt = (CASEDIR / case / 'CaseDocs' / fname).read_text()
    return re.search(rf"\b{key}\s*=\s*(.*?)\n\s*(?:\w+\s*=|/)", txt, re.S).group(1).strip()

def xml_value(case, key):
    for f in sorted((CASEDIR / case).glob('env_*.xml')):
        m = re.search(rf'id="{key}" value="([^"]*)"', f.read_text())
        if m:
            return m.group(1)
    raise KeyError(key)

def yearly(da, years):
    """Monthly means -> calendar-year means weighted by days per month; dims (year, lat, lon)."""
    w = da.time.dt.days_in_month
    out = (da * w).groupby('time.year').sum('time') / w.groupby('time.year').sum('time')
    assert set(years) <= set(out.year.values.tolist()), (years, out.year.values)
    return out.sel(year=list(years))

def hist_files(case, comp, years):
    """Raw monthly h0 files of `years`, from the first archive that has all of them."""
    tag = 'clm2' if comp == 'lnd' else 'cam'
    for root in ARCHIVES:
        fs = [root / case / comp / 'hist' / f'{case}.{tag}.h0.{y}-{m:02d}.nc' for y in years for m in range(1, 13)]
        if all(f.is_file() for f in fs):
            return fs
    raise FileNotFoundError(f'{case} {comp} h0 {years[0]}–{years[-1]}')

# --- MEGAN: CAM species <- CLM compounds (megan_specifier in drv_flds_in) ---
def megan_groups(case):
    spec = re.findall(r"'([^']*)'", read_nml(case, 'megan_specifier', 'drv_flds_in'))
    return {k.strip(): [c.strip() for c in v.split('+')] for k, v in (s.split('=') for s in spec)}

def megan_yearly(alias, case, years):
    """kg m-2 s-1 per grid-cell area: sum of the CLM compounds of each CAM species x landfrac."""
    with xr.open_dataset(PROC / f'{alias}_lnd_BVOC_{PROC_YEARS}.nc') as ds:
        assert ds.attrs['casename'] == case, (alias, ds.attrs['casename'])
        lf = ds.landfrac.fillna(0)
        lf = lf.isel(time=0) if 'time' in lf.dims else lf
        out = {sp: yearly(sum(ds[f'MEG_{c}'].fillna(0) for c in comps) * lf, years)
               for sp, comps in megan_groups(case).items()}
    return xr.Dataset(out).assign_coords(lat=AREA.lat, lon=AREA.lon).load()

# --- dust: CLM DSTFLXT x landfrac x erodibility / dust_emis_fact x 1.15 (oslo_aero_dust_emis) ---
def _dust_month(args):
    path, fac = args
    with netCDF4.Dataset(path) as d:
        return np.ma.filled(d['DSTFLXT'][0], 0.) * np.ma.filled(d['landfrac'][:], 0.) * fac

def dust_yearly(alias, case, years):
    f = CACHE / f'dust_{alias}_{years[0]}-{years[-1]}.nc'
    if f.is_file() and not REBUILD_CACHE:
        da = xr.open_dataarray(f).load()
        assert np.allclose(da.lat, AREA.lat) and np.allclose(da.lon, AREA.lon)
        return da.assign_coords(lat=AREA.lat, lon=AREA.lon)  # same grid; avoid an outer join on float coords
    erod = xr.open_dataset(read_nml(case, 'soil_erod_file').strip("'")).mbl_bsn_fct_geo
    assert erod.shape == AREA.shape and np.allclose(erod.lat[1:-1], AREA.lat[1:-1])  # same grid (file poles at ±89.5)
    fac = np.where(erod.values < 0.1, 0., erod.values) / float(read_nml(case, 'dust_emis_fact').replace('D0', '')) * 1.15
    files = hist_files(case, 'lnd', years)
    with ProcessPoolExecutor(N_WORKERS) as ex:
        mon = np.stack(list(ex.map(_dust_month, [(p, fac) for p in files]))).reshape(len(years), 12, *AREA.shape)
    da = xr.DataArray((mon * DPM[None, :, None, None]).sum(1) / DPM.sum(), dims=('year', 'lat', 'lon'),
                      coords=dict(year=list(years), lat=AREA.lat, lon=AREA.lon), name='dust',
                      attrs=dict(units='kg m-2 s-1', case=case))
    da.to_netcdf(f)
    return da

# --- ocean: DMS flux and sea-salt number flux, offline from monthly means with the model formulas ---
def ocean_yearly(alias, case, years):
    """oslo_aero_ocean.F90 (DMS 'lana', HAMOCC transfer velocity) and oslo_aero_seasalt.F90 (Salter et al. 2015).
    Inputs: model U10, OCNFRAC, ICEFRAC (monthly), prescribed SST and seawater DMS (monthly climatologies)."""
    with xr.open_dataset(PROC / f'{alias}_atm_METEO_{PROC_YEARS}.nc') as ds:
        assert ds.attrs['casename'] == case, (alias, ds.attrs['casename'])
        m = ds[['U10', 'OCNFRAC', 'ICEFRAC']].load()
    with xr.open_dataset(xml_value(case, 'SSTICE_DATA_FILENAME'), decode_times=False) as s:
        sst = s.SST_cpl_prediddle.load()   # deg C
    with xr.open_dataset(DMS_DIR / read_nml(case, 'ocean_filename').strip("'"), decode_times=False) as s:
        dms = s.dms.fillna(0).load()       # nmol L-1
    for x in (sst, dms):
        assert x.sizes['time'] == 12 and np.allclose(x.lat, m.lat) and np.allclose(x.lon, m.lon)
    idx = m.time.dt.month.values - 1
    like = lambda x: xr.DataArray(x.values[idx], dims=m.U10.dims, coords=m.U10.coords)
    t, c, u = like(sst), like(dms), m.U10
    open_ = (m.OCNFRAC * (1 - m.ICEFRAC)).clip(0, 1)
    sc = 2855.7 + (-177.63 + (6.0438 + (-0.11645 + 0.00094743 * t) * t) * t) * t   # Schmidt number of DMS
    fdms = (MW_DMS * 6.97e-7 * open_ * u**2 * np.sqrt(660 / sc) * 1e-9 * c).fillna(0)  # kg DMS m-2 s-1
    wc = 2e-8 * u**3.74 * open_                                                        # whitecap fraction
    out = dict(DMS_flux=yearly(fdms, years), open_ocean=yearly(open_, years))
    for n, (a, b, cc, d) in enumerate(SS_COEF, 1):
        out[f'seasalt_n{n}'] = yearly((wc * (((a * t + b) * t + cc) * t + d)).fillna(0).clip(min=0), years)  # m-2 s-1
    return xr.Dataset(out).assign_coords(lat=AREA.lat, lon=AREA.lon).load()

# --- lightning NOx: CAM prints the global value every time step in atm.log ---
def lightning_yearly(case, years):
    f = CACHE / f'lightning_{case}.csv'
    if f.is_file() and not REBUILD_CACHE:
        s = pd.read_csv(f, index_col=0).iloc[:, 0]
    else:
        steps = int(xml_value(case, 'ATM_NCPL')) * 365
        logs = {}
        for root in ARCHIVES:
            for g in sorted((root / case / 'logs').glob('atm.log.*.gz')):
                logs.setdefault(g.name, g)
        out = {}
        for g in logs.values():
            y0, v = None, []
            with gzip.open(g, 'rt', errors='replace') as fh:
                for line in fh:
                    if 'lightning NOx (TgN/y)' in line:
                        v.append(float(line.rsplit('=', 1)[1].split()[1]))
                    elif y0 is None and 'Current date (yr mon day tod)' in line:
                        y0 = int(line.split(':')[1].split()[0])
            if len(v) % steps == 1:   # the first segment also prints step 0
                v = v[1:]
            assert len(v) % steps == 0, (g.name, len(v))
            for k in range(len(v) // steps):
                assert y0 + k not in out, (g.name, y0 + k)
                out[y0 + k] = float(np.mean(v[k * steps:(k + 1) * steps]))
        s = pd.Series(out, name='lightning NOx (Tg N/yr)').sort_index()
        s.to_csv(f)
    missing = set(years) - set(s.index)
    assert not missing, (case, sorted(missing))
    return s.loc[list(years)]


In [ ]:
SRC, LIGHT = {}, {}
for p in CASES:
    meg = megan_yearly(ALIAS[p], CASES[p], YEARS)
    ocn = ocean_yearly(ALIAS[p], CASES[p], YEARS)
    SRC[p] = xr.merge([meg, ocn, dust_yearly(ALIAS[p], CASES[p], YEARS).rename('dust')])
    LIGHT[p] = lightning_yearly(CASES[p], YEARS)

# check: CLM compounds x landfrac reproduce what CAM receives from MEGAN (atm MEG_ISOP, MEG_MTERP)
for p in CASES:
    with xr.open_dataset(PROC / f'{ALIAS[p]}_atm_BVOC_{PROC_YEARS}.nc') as ds:
        for sp in ('ISOP', 'MTERP'):
            cam = yearly(ds[f'MEG_{sp}'], YEARS).assign_coords(lat=AREA.lat, lon=AREA.lon)
            r = float((SRC[p][sp] * AREA).sum() / (cam * AREA).sum())
            print(f'{p} {sp}: CLM-based / CAM MEG_{sp} = {r:.5f}')
            assert abs(r - 1) < 1e-3

MEG_OTHER = [v for v in megan_groups(CASES['PD']) if v not in ('ISOP', 'MTERP', 'BCARY', 'CH3OH')]
SRC_VARS = {  # name: (function of the yearly dataset -> kg or # m-2 s-1, factor to the total unit, unit)
    'MEGAN isoprene': (lambda d: d.ISOP, 1e-9, 'Tg yr⁻¹'),
    'MEGAN monoterpenes': (lambda d: d.MTERP, 1e-9, 'Tg yr⁻¹'),
    'MEGAN sesquiterpenes': (lambda d: d.BCARY, 1e-9, 'Tg yr⁻¹'),
    'MEGAN methanol': (lambda d: d.CH3OH, 1e-9, 'Tg yr⁻¹'),
    'MEGAN other (15 species)': (lambda d: sum(d[v] for v in MEG_OTHER), 1e-9, 'Tg yr⁻¹'),
    'dust': (lambda d: d.dust, 1e-9, 'Tg yr⁻¹'),
    'DMS flux (offline)': (lambda d: d.DMS_flux * MW_S / MW_DMS, 1e-9, 'Tg S yr⁻¹'),
    'sea salt, mode 1 (offline)': (lambda d: d.seasalt_n1, 1e-27, '10²⁷ yr⁻¹'),
    'sea salt, mode 2 (offline)': (lambda d: d.seasalt_n2, 1e-27, '10²⁷ yr⁻¹'),
    'sea salt, mode 3 (offline)': (lambda d: d.seasalt_n3, 1e-27, '10²⁷ yr⁻¹'),
    'open-ocean area': (lambda d: d.open_ocean / SEC_YR, 1e-12, '10⁶ km²'),
}
SRC_REG = {'global': AREA, '46–82°N': REGIONS['46–82°N'], 'Arctic >66.5°N': REGIONS['Arctic >66.5°N']}
rows = []
for name, (fn, fac, unit) in SRC_VARS.items():
    for r, w in SRC_REG.items():
        v = {p: float((fn(SRC[p]).mean('year') * w).sum()) * SEC_YR * fac for p in CASES}
        rows.append(dict(source=name, unit=unit, region=r, PD=v['PD'], FUT=v['FUT'],
                         **{'FUT − PD': v['FUT'] - v['PD'], 'change %': 100 * (v['FUT'] / v['PD'] - 1) if v['PD'] else np.nan}))
for r in ('global',):
    v = {p: LIGHT[p].mean() for p in CASES}
    rows.append(dict(source='lightning NOx', unit='Tg N yr⁻¹', region=r, PD=v['PD'], FUT=v['FUT'],
                     **{'FUT − PD': v['FUT'] - v['PD'], 'change %': 100 * (v['FUT'] / v['PD'] - 1)}))
SRC_TAB = pd.DataFrame(rows).set_index(['source', 'unit', 'region'])
SRC_TAB.to_csv(TAB / f'online_sources_ctrl_pd_fut_{PROC_YEARS}.csv')
pd.DataFrame(LIGHT).rename_axis('year').to_csv(TAB / 'lightning_nox_yearly.csv')
with pd.option_context('display.float_format', '{:.3g}'.format, 'display.max_rows', 100):
    display(SRC_TAB.xs('global', level='region'))
    display(SRC_TAB.xs('46–82°N', level='region'))

In [ ]:
# lightning NOx per year
fig, ax = plt.subplots(figsize=(7, 3))
for p in CASES:
    ax.plot(LIGHT[p].index, LIGHT[p].values, 'o-', color=PCOL[p], ms=3, label=f'{ALIAS[p]} (mean {LIGHT[p].mean():.2f})')
ax.set_ylabel('Tg N yr⁻¹'); ax.set_title('Global lightning NOx (from atm.log, every time step)')
ax.set_ylim(bottom=0); ax.legend(frameon=False); ax.spines[['top', 'right']].set_visible(False)
fig.savefig(FIG / 'lightning_nox_yearly.png', dpi=200, bbox_inches='tight'); plt.show()

# maps of the online sources, 2001–2025 mean (Arctic; global in the Appendix)
lab = {p: f'{ALIAS[p]}, {YRS}' for p in CASES}
GLOB = SRC_TAB.xs('global', level='region')
for name, unit, fn, fname in [
        ('MEGAN isoprene', 'g m⁻² yr⁻¹', lambda d: d.ISOP * SEC_YR * 1e3, 'online_map_megan_isoprene.png'),
        ('MEGAN monoterpenes', 'g m⁻² yr⁻¹', lambda d: d.MTERP * SEC_YR * 1e3, 'online_map_megan_monoterpenes.png'),
        ('dust', 'g m⁻² yr⁻¹', lambda d: d.dust * SEC_YR * 1e3, 'online_map_dust.png'),
        ('DMS flux (offline)', 'g S m⁻² yr⁻¹', lambda d: d.DMS_flux * MW_S / MW_DMS * SEC_YR * 1e3, 'online_map_dms_flux.png'),
        ('sea salt, mode 2 (offline)', '10¹² m⁻² yr⁻¹', lambda d: d.seasalt_n2 * SEC_YR * 1e-12, 'online_map_seasalt_mode2.png')]:
    g = GLOB.loc[name]
    triptych({p: fn(SRC[p]).mean('year') for p in CASES},
             f'{name} (global: PD {g.PD.iloc[0]:.3g}, FUT {g.FUT.iloc[0]:.3g} {g.index[0]})', unit, fname, labels=lab)

### 2.1 Totals and time series of the online sources: global and north of 40°N

Same two figures as for the prescribed emissions (§3 and §6 of the prescribed notebook), for the online sources. **Bars** = global total, mean of the 25 years 2001–2025 (error bar = ±1 standard deviation of the yearly totals), stacked by component. **Time series** = global total of each year; **solid = CTRL_PD**, **dashed with dots = CTRL_FUT**; gold = sum of the components. MEGAN components as in the table above; sea salt = number flux of the three modes; DMS flux and sea salt offline from monthly means (see the table of §2).

The last two figures are the same **north of 40°N** (cells with centre ≥ 40°N; `LAT_BOX`). Lightning NOx is global only (CAM prints one global value in `atm.log`), so it has no panel there.


In [ ]:
ONLINE = {  # panel -> {component: (function of SRC[p] -> kg or # m-2 s-1, factor to the panel unit)}
    'MEGAN BVOC (Tg)': {'isoprene': (lambda d: d.ISOP, 1e-9), 'monoterpenes': (lambda d: d.MTERP, 1e-9),
                        'sesquiterpenes': (lambda d: d.BCARY, 1e-9), 'methanol': (lambda d: d.CH3OH, 1e-9),
                        'other 15 species': (lambda d: sum(d[v] for v in MEG_OTHER), 1e-9)},
    'dust (Tg)': {'dust': (lambda d: d.dust, 1e-9)},
    'DMS flux, offline (Tg S)': {'DMS flux': (lambda d: d.DMS_flux * MW_S / MW_DMS, 1e-9)},
    'sea salt number, offline (10²⁷)': {f'sea salt mode {n}': (lambda d, n=n: d[f'seasalt_n{n}'], 1e-27) for n in (1, 2, 3)},
}
# yearly totals: {(period, panel): DataFrame(year x component)}, global and north of LAT_BOX
def online_totals(w):
    return {(p, panel): pd.DataFrame({c: (fn(SRC[p]) * w).sum(('lat', 'lon')).to_series() * SEC_YR * fac
                                      for c, (fn, fac) in comps.items()})
            for p in CASES for panel, comps in ONLINE.items()}
ONL = online_totals(AREA)
ONL_N = online_totals(AREA.where(AREA.lat >= LAT_BOX, 0))
for p in CASES:
    ONL[p, 'lightning NOx (Tg N)'] = pd.DataFrame({'lightning NOx': LIGHT[p]})  # global only (atm.log)
PANELS = list(ONLINE) + ['lightning NOx (Tg N)']
PANELS_N = list(ONLINE)
COMPCOL = dict(zip(['isoprene', 'monoterpenes', 'sesquiterpenes', 'methanol', 'other 15 species'], [TEAL, NAVY, PLUM, RED, ORANGE]))
COMPCOL.update({'sea salt mode 1': TEAL, 'sea salt mode 2': NAVY, 'sea salt mode 3': PLUM,
                'dust': TEAL, 'DMS flux': TEAL, 'lightning NOx': TEAL})
for p in CASES:
    assert all(list(ONL[p, k].index) == YEARS for k in PANELS)
pd.concat({f'{ALIAS[p]}|{k}': ONL[p, k] for p in CASES for k in PANELS}, names=['run|panel', 'year']).to_csv(
    TAB / f'online_sources_global_yearly_{PROC_YEARS}.csv')
pd.concat({f'{ALIAS[p]}|{k}': ONL_N[p, k] for p in CASES for k in PANELS_N}, names=['run|panel', 'year']).to_csv(
    TAB / f'online_sources_north{LAT_BOX}N_yearly_{PROC_YEARS}.csv')
# check: the 25-year mean of the yearly totals = the global row of the table above
assert np.isclose(ONL['PD', 'MEGAN BVOC (Tg)'].isoprene.mean(), SRC_TAB.loc[('MEGAN isoprene', 'Tg yr⁻¹', 'global'), 'PD'], rtol=1e-6)
assert np.isclose(ONL['FUT', 'dust (Tg)'].dust.mean(), SRC_TAB.loc[('dust', 'Tg yr⁻¹', 'global'), 'FUT'], rtol=1e-6)

# --- bars: 2001-2025 mean, stacked by component ---
def online_bars(ONL, PANELS, where, fname):
    fig, axs = plt.subplots(1, len(PANELS), figsize=(17, 3.8), gridspec_kw=dict(wspace=0.55))
    for ax, panel in zip(axs, PANELS):
        bottom = np.zeros(2)
        for c in ONL['PD', panel].columns:
            v = np.array([ONL[p, panel][c].mean() for p in CASES])
            ax.bar([0, 1], v, bottom=bottom, color=COMPCOL[c], edgecolor='white', lw=0.5, label=c)
            bottom += v
        sd = [ONL[p, panel].sum(axis=1).std(ddof=1) for p in CASES]
        ax.errorbar([0, 1], bottom, yerr=sd, fmt='none', ecolor='k', elinewidth=0.8, capsize=3)
        for x, (v, e) in enumerate(zip(bottom, sd)):
            ax.text(x, v + e, f'{float(f"{v:.3g}"):g}', ha='center', va='bottom', fontsize=8)
        ax.set_xticks([0, 1], [f'{ALIAS[p]}' for p in CASES], fontsize=8)
        ax.set_title(panel, fontsize=10)
        ax.set_ylim(0, (bottom + sd).max() * 1.15)
        ax.spines[['top', 'right']].set_visible(False)
    h, l = [], []
    for ax in axs:
        for hh, ll in zip(*ax.get_legend_handles_labels()):
            if ll not in l and ll not in ('dust', 'DMS flux', 'lightning NOx'):
                h.append(hh); l.append(ll)
    fig.legend(h, l, loc='upper center', ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.06))
    fig.suptitle(f'Online sources: {where} total per year, mean {YRS} (error bar = ±1 s.d. of the yearly totals)', y=1.02)
    fig.savefig(FIG / fname, dpi=200, bbox_inches='tight')
    plt.show()

# --- time series of the yearly global totals ---
def online_series(ONL, PANELS, where, fname):
    fig, axs = plt.subplots(2, 3, figsize=(17, 7.5))
    for ax in axs.flat[len(PANELS):-1]:
        ax.axis('off')
    for ax, panel in zip(axs.flat, PANELS):
        for p, ls, mk in [('PD', '-', None), ('FUT', '--', 'o')]:
            s = ONL[p, panel]
            multi = s.shape[1] > 1
            for c in s.columns:
                ax.plot(s.index, s[c].values, ls=ls, marker=mk, ms=2.5, lw=1, color=COMPCOL[c] if multi else GOLD,
                        label=(c if multi else 'total') if p == 'PD' else None)
            if multi:
                ax.plot(s.index, s.sum(axis=1).values, ls=ls, marker=mk, ms=3, lw=2.2, color=GOLD,
                        label='total (sum of components)' if p == 'PD' else None)
        ax.set_title(panel, fontsize=10)
        ax.set_xlim(YEARS[0] - 0.5, YEARS[-1] + 0.5)
        ax.set_xticks(range(2001, 2026, 4))
        ax.set_ylim(bottom=0)
        ax.spines[['top', 'right']].set_visible(False)
    h, l = [], []
    for ax in axs.flat[:len(PANELS)]:
        for hh, ll in zip(*ax.get_legend_handles_labels()):
            if ll not in l and ll != 'total':
                h.append(hh); l.append(ll)
    h += [plt.Line2D([], [], color='k', ls='-'), plt.Line2D([], [], color='k', ls='--', marker='o', ms=3)]
    l += [ALIAS['PD'], ALIAS['FUT']]
    axs.flat[-1].axis('off')
    axs.flat[-1].legend(h, l, loc='center', frameon=False, fontsize=9)
    fig.suptitle(f'Online sources: {where} total of each year, {YRS}; solid = CTRL_PD, dashed = CTRL_FUT (single-component panels: gold)')
    fig.tight_layout()
    fig.savefig(FIG / fname, dpi=200, bbox_inches='tight')
    plt.show()

online_bars(ONL, PANELS, 'global', 'online_totals_mean.png')
online_series(ONL, PANELS, 'global', 'online_timeseries.png')
online_bars(ONL_N, PANELS_N, f'north of {LAT_BOX}°N', f'online_totals_mean_north{LAT_BOX}N.png')
online_series(ONL_N, PANELS_N, f'north of {LAT_BOX}°N', f'online_timeseries_north{LAT_BOX}N.png')


## 3. Summary

Tags: **(a)** read from the tables above · **(b)** inference. The prescribed inputs (emissions of the year read, ocean DMS concentration, greenhouse gases) are summarised in `background_aerosol_pd_fut_prescribed.ipynb`.

**What the model makes of it (CTRL_PD → CTRL_FUT, 2001–2025)** (a)

| | global | 46–82°N (forest-expansion band) | Arctic >66.5°N |
|---|---|---|---|
| sulfate burden | 6.23 → 3.78 mg m⁻² (−39 %) | −46 % | −41 % (7.89 → 4.69) |
| surface SO₂ | −67 % | −84 % | −75 % |
| surface NOx | −36 % | −50 % | +5 % |
| surface O₃ | +3 % (27.9 → 28.8 ppb) | −1 % | −2 % |
| surface CO | +30 % | +8 % | +13 % |
| surface DMS | +35 % | **+177 %** | **+413 %** |
| droplet column | −9 % | −14 % | **+25 %** |
| AOD 550 nm | **+6 %** (0.203 → 0.216) | −16 % | −17 % |
| absorption AOD | −9 % | −24 % | −35 % |

The 2001–2025 values differ from the earlier 2001–2014 ones by at most ~2 % (e.g. FUT surface DMS at 46–82°N 136 → 138 ppt); percent changes move by ≤ 1 point (old tables in `background_aerosol_pd_fut_online_outputs/2001-2014/`).

Inferences (b):
- Global AOD goes **up** although sulfate falls by 39 %. The increase sits over the Sahara/Arabia and the tropical oceans, which points to more dust and sea salt (and water uptake) in the warmer FUT climate. This is not verified: the output has no AOD per component (the `A550_*` fields are absorption).
- Surface DMS rises although seawater DMS falls. Likely causes: less sea ice (more open ocean in the Arctic, hence +176 % at 46–82°N) and slower oxidation with less NOx. Not checked.
- CO rises although CO emissions fall: probably the much higher CH₄ of SSP5-8.5 in 2100 (CH₄ oxidation makes CO). Not checked here.
- Ozone barely changes: lower NOx and higher CH₄/CO roughly cancel.

**Online sources (§2), with the greenhouse-gas boundary condition (prescribed notebook §7)** (a)

| | PD | FUT | change |
|---|---|---|---|
| surface CH₄ (boundary condition) | 1778 ppb | 2415 ppb | +36 % |
| surface N₂O / CO₂ | 316 ppb / 369 ppm | 392 ppb / 1135 ppm | +24 % / ×3.1 |
| MEGAN isoprene | 509 Tg yr⁻¹ | 885 | +74 % |
| MEGAN monoterpenes | 143 Tg yr⁻¹ | 510 | **×3.6** |
| MEGAN sesquiterpenes / methanol | 13 / 104 | 57 / 297 | ×4.3 / ×2.9 |
| dust | 3.8 Pg yr⁻¹ | 4.0 | +6 % (46–82°N +30 %) |
| DMS flux (offline) | 16.5 Tg S yr⁻¹ | 15.7 | −5 % globally, **+43 % at 46–82°N** |
| sea salt, number (offline) | | | −9 % (mode 1), −5 %, 0 % |
| open-ocean area 46–82°N | 22.0 × 10⁶ km² | 26.6 | +21 % (less sea ice) |
| lightning NOx | 3.6 Tg N yr⁻¹ | 4.9 | +36 % |

- The biggest change in the natural background is **MEGAN**: in FUT the land emits 3.6× more monoterpenes, the main SOA precursor. So the FUT background has **much less anthropogenic sulfate but much more biogenic SOA precursor**. (a)
- **CH₄ +36 %** explains at least part of the CO rise (+30 %) and the near-constant O₃. (b)
- **DMS:** at 46–82°N the flux rises by 43 % (21 % more open ocean, plus warmer water), yet surface DMS rises by 177 % (+413 % north of 66.5°N). The rest probably comes from slower oxidation (OH, NO₃). Not checked. (b)
- **NH₃ +48 %** has no aerosol of its own in OsloAero but raises cloud-water pH, which speeds up in-cloud SO₂ + O₃ → sulfate (`mo_setsox.F90`). This can partly offset the SO₂ cut; its size is not diagnosed. (a: mechanism; b: relevance)
- The dust total is reconstructed from CLM `DSTFLXT` with the CAM scaling; the absolute value (3.8–4.0 Pg yr⁻¹) is not checked against a model diagnostic. The FUT/PD ratio is what counts here. (b)

**Why it matters for the paper** (b)
1. **Fewer competing particles.** With 39–46 % less sulfate (−84 % surface SO₂ at 46–82°N), FUT has fewer pre-existing particles and fewer CCN. New SOA (from more BVOC) then forms or grows particles in a cleaner air mass, where each extra CCN changes cloud droplets more (cloud susceptibility is higher at low droplet number). The **same BVOC change can have a larger aerosol–cloud effect in FUT than in PD.**
2. **A larger natural share.** At high latitudes more of the remaining sulfate comes from DMS (more open ocean) and volcanoes. DMS also makes some SOA precursors in this mechanism. So the forest's BVOC adds to a background that is **more marine and more biogenic**, not to a polluted one.
3. **Higher BVOC baseline.** FUT CTRL already emits 3.6× more monoterpenes than PD. An extra forest source is a smaller *relative* addition in FUT, and SOA formation may saturate (less oxidant per BVOC; more condensation surface from SOA itself).
4. **Implication.** Points 1 and 3 push in opposite directions, so the size of the FUT forest-expansion aerosol effect cannot be guessed from PD. It has to come from the runs (`background_aerosol_lcc_ctrl_fut_online.ipynb`: very small).

**For the paper:** FUT is a **clean-air background** for anthropogenic aerosol (sulfate, BC, anthropogenic OM, NOx), with unchanged fires and volcanoes. BVOC-driven SOA formation and aerosol–cloud effects of the forest expansion therefore act on a lower sulfate/CCN background in FUT. (b)


**Arctic (b, not checked):** in the Arctic the droplet column goes **up** by 25 % although sulfate falls by 41 %, unlike the global and 46–82°N means. It coincides with the large DMS rise (more open ocean) and possibly more cloud/sea-salt over newly ice-free water; not separated here.

## Appendix: global maps

The same maps as in §1–2, over the whole globe (Robinson projection). Colour levels of the §2 maps are recomputed from the global fields. Files in `figures/appendix-global/`.

In [ ]:
for fn, args, kw in list(APPENDIX):
    fn(*args, region='global', **kw)
